In [40]:
import importlib
from pathlib import Path
import utils_database

importlib.reload(utils_database)

from utils_database import *

In [12]:
df = pd.read_excel("24ACE-FH-EARLY_DHM_v105_20260424.xls", sheet_name="Data Handling Manual FH-EARLY")
variables = df[df["Référence"].notna()].copy()

variables = variables[["Id", "Référence", "Saisie", "Type", "Format", "Intitulé"]]

In [13]:
export_folder = Path("export-example")

inc = read_export_csv(export_folder / "INC_ANSI.csv")
ml = read_export_csv(export_folder / "ML_ANSI.csv")
tab_bs = read_export_csv(export_folder / "TAB_BS_ANSI.csv")
tab_fh = read_export_csv(export_folder / "TAB_FH_ANSI.csv")

export_tables = {"INC": inc, "ML": ml, "TAB_BS": tab_bs, "TAB_FH": tab_fh}

In [14]:
for var in ["SEX", "BRTHDAT", "AGE"]:
    print(var, "->", find_export_columns(var, inc.columns))

# ml columns
print("FH7ALS ->", find_export_columns("FH7ALS", ml.columns))

SEX -> ['SEX']
BRTHDAT -> ['BRTHDAT_D', 'BRTHDAT_M', 'BRTHDAT_Y', 'BRTHDAT']
AGE -> ['AGE', 'AGE_V']
FH7ALS -> ['FH7ALS_C1', 'FH7ALS_C2', 'FH7ALS_C3', 'FH7ALS_C4', 'FH7ALS_C5', 'FH7ALS_C6', 'FH7ALS_C7', 'FH7ALS_C8', 'FH7ALS_C9', 'FH7ALS_C10', 'FH7ALS_C11', 'FH7ALS_C12', 'FH7ALS_C13', 'FH7ALS_C14']


In [15]:
export_tables = {"INC": inc, "ML": ml, "TAB_BS": tab_bs, "TAB_FH": tab_fh,}

mapped = create_dictionary_export_map(variables, export_tables)
summary, mismatched = summarize_dictionary_export_map(mapped)

In [16]:
mapped["Type_english"] = mapped["Type"].apply(classify_variable_type)
mapped[["Référence", "Type", "Type_english", "Intitulé", "INC_export_cols", "ML_export_cols", "TAB_BS_export_cols", "TAB_FH_export_cols"]].head()

,Référence,Type,Type_english,Intitulé,INC_export_cols,ML_export_cols,TAB_BS_export_cols,TAB_FH_export_cols
2,PROJECTNAME,Label,label,FH-Early,[],[],[],[]
3,SEX,Radio buttons,categorical,Gender,[SEX],[],[],[]
4,BRTHDAT,Date,date,Date of birth,"[BRTHDAT_D, BRTHDAT_M, BRTHDAT_Y, BRTHDAT]",[],[],[]
5,AGE,Entier,numeric,Age,"[AGE, AGE_V]",[],[],[]
6,INVNAME,Texte,text,Investigator,[INVNAME],[],[],[]


In [18]:
mapped["code_labels"] = mapped["Format"].apply(parse_code_labels)
mapped[["Référence", "Type", "Format", "code_labels"]].head(20)

,Référence,Type,Format,code_labels
2,PROJECTNAME,Label,NaN,{}
3,SEX,Radio buttons,1 = Male \n2 = Female,"{1: 'Male', 2: 'Female'}"
4,BRTHDAT,Date,Précision = MM/YYYY\nValeurs par défaut = 01/0...,{}
5,AGE,Entier,Taille : N\nMin = 18\nMax = 120,{}
6,INVNAME,Texte,Taille : N,{}
7,SITEID,Texte,Taille : N,{}
8,SUBJECT_REF,Texte,Taille : N,{}
9,RFICDAT,Date,Précision = JJ/MM/YYYY\nValeurs par défaut = 0...,{}
12,INF1YN,Radio buttons,1 = Yes \n0 = No,"{1: 'Yes', 0: 'No'}"
13,INF1AYN,Radio buttons,1 = Yes \n0 = No,"{1: 'Yes', 0: 'No'}"


In [ ]:
# check that code labels are captured for all categorical and checkbox variables
mapped[
    (mapped["Type_english"].isin(["categorical", "checkbox"])) &
    (mapped["code_labels"].apply(len) == 0)
][["Référence", "Type", "Format", "Intitulé"]]

,Référence,Type,Format,Intitulé


In [36]:
# mapped.to_csv("dictionary_export_map.csv", index=False)
mapped.columns

Index(['Id', 'Référence', 'Saisie', 'Type', 'Format', 'Intitulé',
       'INC_export_cols', 'ML_export_cols', 'TAB_BS_export_cols',
       'TAB_FH_export_cols', 'n_export_matches', 'has_export_match',
       'Type_english', 'code_labels'],
      dtype='object')

In [20]:
mapped[
    ["Référence", "Type", "Type_english", "Intitulé",
     "code_labels", "has_export_match", "n_export_matches"]
].head(20)

,Référence,Type,Type_english,Intitulé,code_labels,has_export_match,n_export_matches
2,PROJECTNAME,Label,label,FH-Early,{},False,0
3,SEX,Radio buttons,categorical,Gender,"{1: 'Male', 2: 'Female'}",True,1
4,BRTHDAT,Date,date,Date of birth,{},True,4
5,AGE,Entier,numeric,Age,{},True,2
6,INVNAME,Texte,text,Investigator,{},True,1
7,SITEID,Texte,text,Centre,{},True,1
8,SUBJECT_REF,Texte,text,Patient ID,{},True,5
9,RFICDAT,Date,date,Date of inclusion,{},True,4
12,INF1YN,Radio buttons,categorical,Patient deceased at the time of inclusion,"{1: 'Yes', 0: 'No'}",True,1
13,INF1AYN,Radio buttons,categorical,"If so, did you verify that there was no object...","{1: 'Yes', 0: 'No'}",True,1


In [22]:
sas_labels = parse_all_sas_labels(export_folder)

sas_labels.keys()

dict_keys(['ML', 'TAB_FH', 'TAB_BS', 'INC'])

In [29]:
sas_informats = parse_all_sas_informats(export_folder)

column_catalog = create_export_column_catalog(
    export_tables,
    sas_labels,
    sas_informats
)

column_catalog.head(-30)

,table,column,sas_label,sas_informat
0,INC,STUDY_ID,Identification étude,$8.
1,INC,COUNTRY_ID,Identifiant pays,$4.
2,INC,EXTRACTION_DATE,Date extraction,$19.
3,INC,SITE_ID,Centre,$4.
4,INC,SUBJECT_ID,Identifiant unique,$4.
...,...,...,...,...
346,TAB_BS,COUNTRY_ID,Identifiant pays,$4.
347,TAB_BS,EXTRACTION_DATE,Date extraction,$19.
348,TAB_BS,SITE_ID,Centre,$4.
349,TAB_BS,SUBJECT_ID,Identifiant unique,$4.


In [30]:
# sanity check to control that the sas catalog is stable
# 1. total rows = total CSV columns
sum(len(t.columns) for t in export_tables.values()), len(column_catalog)

(381, 381)

In [31]:
# 2. no missing SAS labels
column_catalog[column_catalog["sas_label"].isna()]

,table,column,sas_label,sas_informat


In [32]:
# 3. no missing SAS informats
column_catalog[column_catalog["sas_informat"].isna()]

,table,column,sas_label,sas_informat


In [33]:
# 4. check informat types
column_catalog["sas_informat"].value_counts()

sas_informat
$4.          267
best32.       62
DDMMYY10.     16
$5.            9
$8.            6
$9.            5
$19.           4
$7.            3
$6.            2
$63.           1
$12.           1
$21.           1
$49.           1
$45.           1
$50.           1
$244.          1
Name: count, dtype: int64

In [34]:
# 5. sample manually
column_catalog.sample(20, random_state=1)

,table,column,sas_label,sas_informat
244,ML,BS21NUM_V,HbA1c*/ Valeur entière ou réelle,best32.
180,ML,BS1A1TXT,"If etc., specify",$21.
277,ML,BS34YN,Gene score calculation,$4.
185,ML,BS4NUM_V,TAG1/ Valeur entière ou réelle,best32.
233,ML,BS39LS_C5,Treatment at the time of analysis / Lomitapide,$4.
314,ML,BS38DNUM_V,CIMT (Carotid Intima Media Thickness) Left (mm...,best32.
138,ML,MH16J1NUM,Carotid endarterectomy age*/ Valeur au format ...,$4.
201,ML,BS10LS,Lp(a)1 unit,$4.
117,ML,MH5B1NUM_V,PTCA Age/ Valeur entière ou réelle,best32.
301,ML,BS37CNUM,Computed Coronary Tomography Angiography (arte...,$4.


In [43]:
master_catalog = create_master_column_catalog(column_catalog, variables)

master_catalog.sample(20, random_state=1)

,table,column,sas_label,sas_informat,Référence,Id,Saisie,Type,Format,Intitulé
244,ML,BS21NUM_V,HbA1c*/ Valeur entière ou réelle,best32.,BS21NUM,ID102S2V117,O,Réel,"Format : #,#\nMin = 4.0\nMax = 9.9\nUnité = %",HbA1c*
180,ML,BS1A1TXT,"If etc., specify",$21.,BS1A1TXT,ID102S2V96,O,Texte,Taille : N,"If etc., specify"
277,ML,BS34YN,Gene score calculation,$4.,BS34YN,ID102S2V135,O,Radio buttons,1 = Yes \n0 = No,Gene score calculation
185,ML,BS4NUM_V,TAG1/ Valeur entière ou réelle,best32.,BS4NUM,ID102S2V98,O,Réel,"Format : ##,##\nMin = 0.0\nMax = 99.0",TAG1
233,ML,BS39LS_C5,Treatment at the time of analysis / Lomitapide,$4.,BS39LS,ID102S2V231,O,Cases à cocher,00 = Statin \n1 = ezetimibe \n2 = cholestyrami...,Treatment at the time of analysis
314,ML,BS38DNUM_V,CIMT (Carotid Intima Media Thickness) Left (mm...,best32.,BS38DNUM,ID102S2V155,O,Réel,"Format : ###,###\nUnité = mm",CIMT (Carotid Intima Media Thickness) Left (mm)*
138,ML,MH16J1NUM,Carotid endarterectomy age*/ Valeur au format ...,$4.,MH16J1NUM,ID102S2V74,O,Entier,Taille : 2\nMax = 120,Carotid endarterectomy age*
201,ML,BS10LS,Lp(a)1 unit,$4.,BS10LS,ID102S2V106,O,Combo box,1 = nmol/L \n2 = mg/dL,Lp(a)1 unit
117,ML,MH5B1NUM_V,PTCA Age/ Valeur entière ou réelle,best32.,MH5B1NUM,ID102S2V58,O,Entier,Taille : 2\nMax = 120,PTCA Age
301,ML,BS37CNUM,Computed Coronary Tomography Angiography (arte...,$4.,BS37CNUM,ID102S2V149,O,Réel,"Format : ###,N\nMin = 0.0\nMax = 100.0\nUnité = %",Computed Coronary Tomography Angiography (arte...


In [44]:
master_catalog["Type_english"] = master_catalog["Type"].apply(classify_variable_type)
master_catalog["code_labels"] = master_catalog["Format"].apply(parse_code_labels)

In [45]:
master_catalog.sample(20, random_state=1)

,table,column,sas_label,sas_informat,Référence,Id,Saisie,Type,Format,Intitulé,Type_english,code_labels
244,ML,BS21NUM_V,HbA1c*/ Valeur entière ou réelle,best32.,BS21NUM,ID102S2V117,O,Réel,"Format : #,#\nMin = 4.0\nMax = 9.9\nUnité = %",HbA1c*,numeric,{}
180,ML,BS1A1TXT,"If etc., specify",$21.,BS1A1TXT,ID102S2V96,O,Texte,Taille : N,"If etc., specify",text,{}
277,ML,BS34YN,Gene score calculation,$4.,BS34YN,ID102S2V135,O,Radio buttons,1 = Yes \n0 = No,Gene score calculation,categorical,"{1: 'Yes', 0: 'No'}"
185,ML,BS4NUM_V,TAG1/ Valeur entière ou réelle,best32.,BS4NUM,ID102S2V98,O,Réel,"Format : ##,##\nMin = 0.0\nMax = 99.0",TAG1,numeric,{}
233,ML,BS39LS_C5,Treatment at the time of analysis / Lomitapide,$4.,BS39LS,ID102S2V231,O,Cases à cocher,00 = Statin \n1 = ezetimibe \n2 = cholestyrami...,Treatment at the time of analysis,checkbox,"{0: 'Statin', 1: 'ezetimibe', 2: 'cholestyrami..."
314,ML,BS38DNUM_V,CIMT (Carotid Intima Media Thickness) Left (mm...,best32.,BS38DNUM,ID102S2V155,O,Réel,"Format : ###,###\nUnité = mm",CIMT (Carotid Intima Media Thickness) Left (mm)*,numeric,{}
138,ML,MH16J1NUM,Carotid endarterectomy age*/ Valeur au format ...,$4.,MH16J1NUM,ID102S2V74,O,Entier,Taille : 2\nMax = 120,Carotid endarterectomy age*,numeric,{}
201,ML,BS10LS,Lp(a)1 unit,$4.,BS10LS,ID102S2V106,O,Combo box,1 = nmol/L \n2 = mg/dL,Lp(a)1 unit,categorical,"{1: 'nmol/L', 2: 'mg/dL'}"
117,ML,MH5B1NUM_V,PTCA Age/ Valeur entière ou réelle,best32.,MH5B1NUM,ID102S2V58,O,Entier,Taille : 2\nMax = 120,PTCA Age,numeric,{}
301,ML,BS37CNUM,Computed Coronary Tomography Angiography (arte...,$4.,BS37CNUM,ID102S2V149,O,Réel,"Format : ###,N\nMin = 0.0\nMax = 100.0\nUnité = %",Computed Coronary Tomography Angiography (arte...,numeric,{}
